# 2. Exploratory Data Analysis

## Objective
Perform comprehensive exploratory data analysis on the training data and any external data sources.

### Data Sources:
1. Training data (train.csv)
2. External lexicon data (from data augmentation)
3. Publications data (for potential additional context)

In [ ]:
# Parameters and Paths
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re

OUTPUT_DIR = Path("output")
EXTERNAL_DATA_DIR = OUTPUT_DIR / "external_data"
IMAGES_DIR = OUTPUT_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data")
print(f"Output directory: {OUTPUT_DIR}")
print(f"External data directory: {EXTERNAL_DATA_DIR}")
print(f"Images directory: {IMAGES_DIR}")

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 2.1 Load Training Data

In [ ]:
# Load training data
train_df = pd.read_csv(DATA_DIR / "train.csv")
print(f"Training data shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nData types:\n{train_df.dtypes}")
print(f"\nMissing values:\n{train_df.isnull().sum()}")

In [ ]:
# View sample data
print("Sample entries:")
train_df.head(3)

## 2.2 Text Length Analysis

In [ ]:
# Calculate text lengths
train_df['transliteration_len'] = train_df['transliteration'].str.len()
train_df['translation_len'] = train_df['translation'].str.len()
train_df['transliteration_words'] = train_df['transliteration'].str.split().str.len()
train_df['translation_words'] = train_df['translation'].str.split().str.len()

# Summary statistics
print("=== Transliteration Length Statistics ===")
print(train_df[['transliteration_len', 'transliteration_words']].describe())

print("\n=== Translation Length Statistics ===")
print(train_df[['translation_len', 'translation_words']].describe())

In [ ]:
# Create visualization of text lengths
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Transliteration character length
axes[0, 0].hist(train_df['transliteration_len'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Character Length')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Transliteration Character Length Distribution')
axes[0, 0].axvline(train_df['transliteration_len'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['transliteration_len'].mean():.0f}")
axes[0, 0].legend()

# Translation character length
axes[0, 1].hist(train_df['translation_len'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Character Length')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Translation Character Length Distribution')
axes[0, 1].axvline(train_df['translation_len'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['translation_len'].mean():.0f}")
axes[0, 1].legend()

# Transliteration word count
axes[1, 0].hist(train_df['transliteration_words'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Transliteration Word Count Distribution')
axes[1, 0].axvline(train_df['transliteration_words'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['transliteration_words'].mean():.0f}")
axes[1, 0].legend()

# Translation word count
axes[1, 1].hist(train_df['translation_words'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_xlabel('Word Count')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Translation Word Count Distribution')
axes[1, 1].axvline(train_df['translation_words'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['translation_words'].mean():.0f}")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'text_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {IMAGES_DIR / 'text_length_distributions.png'}")

## 2.3 Length Ratio Analysis

In [ ]:
# Calculate length ratio (translation / transliteration)
train_df['length_ratio'] = train_df['translation_len'] / train_df['transliteration_len']
train_df['word_ratio'] = train_df['translation_words'] / train_df['transliteration_words']

print("=== Length Ratio Statistics (Translation / Transliteration) ===")
print(train_df[['length_ratio', 'word_ratio']].describe())

# Scatter plot of length ratios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(train_df['transliteration_len'], train_df['translation_len'], alpha=0.5, s=20)
axes[0].set_xlabel('Transliteration Length (chars)')
axes[0].set_ylabel('Translation Length (chars)')
axes[0].set_title('Transliteration vs Translation Length')

# Add regression line
z = np.polyfit(train_df['transliteration_len'], train_df['translation_len'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, train_df['transliteration_len'].max(), 100)
axes[0].plot(x_line, p(x_line), "r--", alpha=0.8, label=f'y={z[0]:.2f}x+{z[1]:.0f}')
axes[0].legend()

axes[1].hist(train_df['length_ratio'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Length Ratio (Translation / Transliteration)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Length Ratio Distribution')
axes[1].axvline(train_df['length_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['length_ratio'].mean():.2f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'length_ratio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Vocabulary Analysis

In [ ]:
# Analyze vocabulary in transliterations
from collections import Counter

def tokenize_akkadian(text):
    """Tokenize Akkadian transliteration"""
    if pd.isna(text):
        return []
    # Split by whitespace and common delimiters
    tokens = re.findall(r'\b\w+\b|[{}<>.]', str(text))
    return tokens

# Tokenize all transliterations
all_tokens = []
for text in train_df['transliteration']:
    all_tokens.extend(tokenize_akkadian(text))

token_counts = Counter(all_tokens)

print(f"Total tokens: {len(all_tokens)}")
print(f"Unique tokens: {len(token_counts)}")
print(f"\nMost common tokens:")
for token, count in token_counts.most_common(30):
    print(f"  {token}: {count}")

In [ ]:
# Analyze vocabulary in English translations
all_english_tokens = []
for text in train_df['translation']:
    if pd.notna(text):
        words = re.findall(r'\b[a-zA-Z]+\b', str(text).lower())
        all_english_tokens.extend(words)

english_token_counts = Counter(all_english_tokens)

print(f"Total English tokens: {len(all_english_tokens)}")
print(f"Unique English tokens: {len(english_token_counts)}")
print(f"\nMost common English words:")
for token, count in english_token_counts.most_common(30):
    print(f"  {token}: {count}")

In [ ]:
# Visualize vocabulary frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 30 Akkadian tokens
top_akk_tokens = token_counts.most_common(30)
tokens, counts = zip(*top_akk_tokens)
axes[0].barh(range(len(tokens)), counts, color='steelblue')
axes[0].set_yticks(range(len(tokens)))
axes[0].set_yticklabels(tokens)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 30 Akkadian Tokens')

# Top 30 English tokens
top_eng_tokens = english_token_counts.most_common(30)
tokens, counts = zip(*top_eng_tokens)
axes[1].barh(range(len(tokens)), counts, color='coral')
axes[1].set_yticks(range(len(tokens)))
axes[1].set_yticklabels(tokens)
axes[1].invert_yaxis()
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 30 English Tokens')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'vocabulary_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Special Character Analysis

Akkadian transliterations contain special characters and notation that need to be handled.

In [ ]:
# Analyze special characters and patterns in transliterations
def analyze_special_chars(text):
    """Count special characters in Akkadian transliteration"""
    if pd.isna(text):
        return {}
    text = str(text)
    return {
        'gaps': text.count('<gap>') + text.count('<big_gap>'),
        'determinatives': text.count('{'),  # curly brackets for determinatives
        'broken_brackets': text.count('[') + text.count(']'),
        'superscripts': len(re.findall(r'[₀-₉ₓ]', text)),
        'special_markers': text.count('!') + text.count('?') + text.count('/'),
    }

# Apply to all rows
special_chars_df = train_df['transliteration'].apply(analyze_special_chars).apply(pd.Series)
train_df = pd.concat([train_df, special_chars_df], axis=1)

print("=== Special Characters Statistics ===")
print(special_chars_df.describe())

In [ ]:
# Visualize special character distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, col in enumerate(['gaps', 'determinatives', 'broken_brackets', 'superscripts', 'special_markers']):
    ax = axes[idx // 3, idx % 3]
    data = train_df[col]
    ax.hist(data, bins=20, edgecolor='black', alpha=0.7)
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Frequency')
    ax.set_title(f'{col.replace("_", " ").title()} Distribution')
    ax.axvline(data.mean(), color='red', linestyle='--', label=f"Mean: {data.mean():.2f}")
    ax.legend()

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'special_characters_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 Analyze External Lexicon Data

In [ ]:
# Load external lexicon data
lexicon_df = pd.read_csv(DATA_DIR / "OA_Lexicon_eBL.csv")
print(f"Lexicon shape: {lexicon_df.shape}")
print(f"\nColumns: {lexicon_df.columns.tolist()}")
print(f"\nWord types:")
print(lexicon_df['type'].value_counts())

In [ ]:
# Analyze word types
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word type distribution
word_type_counts = lexicon_df['type'].value_counts()
axes[0].bar(word_type_counts.index, word_type_counts.values, color='teal')
axes[0].set_xlabel('Word Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Lexicon Word Type Distribution')
axes[0].tick_params(axis='x', rotation=45)

# Form length distribution
lexicon_df['form_len'] = lexicon_df['form'].str.len()
axes[1].hist(lexicon_df['form_len'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_xlabel('Form Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Lexicon Form Length Distribution')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'lexicon_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Test Data Overview

In [ ]:
# Load test data
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(f"Test data shape: {test_df.shape}")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nSample entries:")
test_df.head()

In [ ]:
# Analyze test data
test_df['transliteration_len'] = test_df['transliteration'].str.len()
test_df['transliteration_words'] = test_df['transliteration'].str.split().str.len()

print("=== Test Data Statistics ===")
print(test_df[['transliteration_len', 'transliteration_words']].describe())

print(f"\nUnique text_ids: {test_df['text_id'].nunique()}")

## 2.8 Compare Train vs Test Data

In [ ]:
# Compare train and test data distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length comparison
axes[0].hist(train_df['transliteration_len'], bins=50, alpha=0.5, label='Train', density=True)
axes[0].hist(test_df['transliteration_len'], bins=20, alpha=0.5, label='Test', density=True)
axes[0].set_xlabel('Transliteration Length (chars)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transliteration Length: Train vs Test')
axes[0].legend()

# Word count comparison
axes[1].hist(train_df['transliteration_words'], bins=50, alpha=0.5, label='Train', density=True)
axes[1].hist(test_df['transliteration_words'], bins=20, alpha=0.5, label='Test', density=True)
axes[1].set_xlabel('Transliteration Word Count')
axes[1].set_ylabel('Density')
axes[1].set_title('Word Count: Train vs Test')
axes[1].legend()

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'train_test_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.9 EDA Summary

In [ ]:
# Create summary statistics
eda_summary = {
    "training_data": {
        "total_samples": len(train_df),
        "avg_transliteration_len": float(train_df['transliteration_len'].mean()),
        "avg_translation_len": float(train_df['translation_len'].mean()),
        "avg_transliteration_words": float(train_df['transliteration_words'].mean()),
        "avg_translation_words": float(train_df['translation_words'].mean()),
        "unique_akkadian_tokens": len(token_counts),
        "unique_english_tokens": len(english_token_counts),
        "avg_length_ratio": float(train_df['length_ratio'].mean()),
    },
    "test_data": {
        "total_samples": len(test_df),
        "unique_documents": int(test_df['text_id'].nunique()),
        "avg_transliteration_len": float(test_df['transliteration_len'].mean()),
    },
    "lexicon": {
        "total_entries": len(lexicon_df),
        "word_types": lexicon_df['type'].value_counts().to_dict()
    }
}

print("=== EDA Summary ===")
for key, value in eda_summary.items():
    print(f"\n{key.upper()}:")
    for k, v in value.items():
        if isinstance(v, dict):
            print(f"  {k}: {v}")
        else:
            print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Save summary
with open(OUTPUT_DIR / "eda_summary.json", 'w') as f:
    json.dump(eda_summary, f, indent=2)
print(f"EDA summary saved to {OUTPUT_DIR / 'eda_summary.json'}")

In [ ]:
# Verify notebook is valid JSON
import json

notebook_path = Path("notebooks/02_exploratory_data_analysis.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")